In [ ]:
import os
import sys

In [ ]:
temp_path = os.getcwd().split('\\')
project_dir = '\\'.join(temp_path[:temp_path.index('SOURCE') + 1])

In [ ]:
sys.path.insert(0, f'{project_dir}/code/Packages')

In [ ]:
import json
import preprocessing
from tqdm import tqdm
from glob import glob

In [ ]:
preprocessing.set_export_folder("final_edgelists")

nodelist = preprocessing.import_nodelist()

Export directory set to G:\Shared drives\DOST FilWordNet x WordSense\Thesis\NetSci Team\documents\THS3\Temp Final Deliverables\SOURCE/data/3 - Network Generation/final_edgelists.


---

# Semset Edgelist Creation

## Import data

In [ ]:
algorithm = 'leiden_mod'

In [ ]:
community_files = glob(f'{project_dir}/data/4 - Word Sense Induction/communities/{algorithm}_filtered_10/*.json')
# community_files

In [ ]:
len(community_files)

4557

In [ ]:
communities = []

for file in tqdm(community_files):
    try:
        with open(file) as f:
            communities.append(json.load(f))
    except:
        print(file)

100%|█████████████████████████████████████████████████████████████████████████████| 4557/4557 [00:16<00:00, 281.39it/s]


In [ ]:
nodelist = preprocessing.get_inverted_nodelist()

## Compute Jaccard Similarity

In [ ]:
def jaccard_similarity(comm1, comm2):
    inter = len(list(set(comm1).intersection(comm2)))
    union = len(set(comm1).union(set(comm2)))
    return float(inter) / union

In [ ]:
communities_masterlist = []
for index, community_file in enumerate(communities):
    word = nodelist[community_file['ego']]
    for community_index, community_content in community_file['community'].items():
        communities_masterlist.append((f'{word}_{community_index}', community_content))

## Create DataFrame

In [ ]:
import pandas as pd

In [ ]:
df = pd.DataFrame()

json_files = []

for idx_1 in tqdm(range(len(communities_masterlist))):
    for idx_2 in range(len(communities_masterlist))[idx_1 + 1:]:
        #print(idx_1, idx_2)
        comm_1 = communities_masterlist[idx_1]
        comm_2 = communities_masterlist[idx_2]
        if comm_1[0].split("_")[0] != comm_2[0].split("_")[0]:
            score = jaccard_similarity(comm_1[1], comm_2[1])
            json_files.append({
                "community_1": comm_1[0],
                "community_2": comm_2[0],
                "weight": score,
            })

 20%|███████████████▎                                                             | 2294/11548 [04:35<18:29,  8.34it/s]


KeyboardInterrupt: 

In [ ]:
json_files

[{'community_1': 'a-list_0', 'community_2': 'aaron_0', 'weight': 0.0},
 {'community_1': 'a-list_0',
  'community_2': 'aaron_1',
  'weight': 0.05128205128205128},
 {'community_1': 'a-list_0', 'community_2': 'aaron_2', 'weight': 0.0},
 {'community_1': 'a-list_0',
  'community_2': 'abala_0',
  'weight': 0.027777777777777776},
 {'community_1': 'a-list_0', 'community_2': 'abala_1', 'weight': 0.0},
 {'community_1': 'a-list_0', 'community_2': 'abala_2', 'weight': 0.0},
 {'community_1': 'a-list_0', 'community_2': 'abala_3', 'weight': 0.0},
 {'community_1': 'a-list_0', 'community_2': 'abalahin_0', 'weight': 0.0},
 {'community_1': 'a-list_0', 'community_2': 'abalahin_1', 'weight': 0.0},
 {'community_1': 'a-list_0',
  'community_2': 'abangan_0',
  'weight': 0.02459016393442623},
 {'community_1': 'a-list_0', 'community_2': 'abangan_1', 'weight': 0.0},
 {'community_1': 'a-list_0', 'community_2': 'abangan_2', 'weight': 0.0},
 {'community_1': 'a-list_0', 'community_2': 'abangan_3', 'weight': 0.0},
 {

In [ ]:
semset_df = pd.DataFrame.from_records(json_files)

In [ ]:
if not os.path.isfile(f'{project_dir}/data/7 - Semset Creation/semset_edgelist_{algorithm}'):
    os.makedirs(f'{project_dir}/data/7 - Semset Creation/semset_edgelist_{algorithm}')

In [ ]:
semset_df.to_csv(f'{project_dir}/data/7 - Semset Creation/semset_edgelist_{algorithm}/semset_edgelist.csv')

# Network Creation

In [ ]:
import igraph as ig
from glob import glob
import pandas as pd

In [ ]:
directory = f'{project_dir}/data/7 - Semset Creation/semset_edgelist_{algorithm}'
filenames = glob(f'{directory}/*.csv')

file_dfs = []
for file in filenames[0:1]:
  file_dfs.append(pd.read_csv(file))

semset_edgelist = pd.concat(file_dfs)[['community_1', 'community_2', 'weight']]

# threshold for filtering those with low weights
#semset_edgelist['weight'] = semset_edgelist['weight'].apply(lambda weight: weight if weight > 0.4 else 0)

In [ ]:
semset_edgelist[(semset_edgelist['weight'] > 0.6) & (semset_edgelist['weight'] < 0.75)]

,community_1,community_2,weight
881892,muse_3,dilaan_1,0.666667
2182611,kalabasa_3,hera_0,0.666667
3919447,karate_3,rugby_4,0.625000
4085005,huwebes_0,martes_0,0.621145
4096108,huwebes_2,biyernes_1,0.625000
4098249,huwebes_2,martes_2,0.633508
4109353,huwebes_4,biyernes_4,0.698413
14167,oslo_2,kenya_3,0.666667
4505671,miyerkules_0,huwebes_0,0.631818
4508367,miyerkules_0,martes_0,0.632558


## Create network


In [ ]:
network = ig.Graph.DataFrame(semset_edgelist, directed=False)
    
# for vertex in network.vs:
#     vertex['word'] = vertex['name']

In [ ]:
network.delete_edges([edge for edge in network.es.select(weight_eq=0)])

In [ ]:
network.write(f"{project_dir}/data/7 - Semset Creation/semset_network.graphml", format="graphml")

In [ ]:
len(network.vs)

11548